In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import subprocess

from pybedtools import BedTool

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
samples = sorted(list(np.loadtxt(f'{projdir}/txt/allc_table/ips_cluster.txt', dtype=str)))
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))

In [ ]:
d = donors[0]
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/mature_ips/early_ips/{d}/{d}_rms_results_collapsed.tsv',
                   header=0, sep='\t')
dmrs.shape

In [ ]:
dmrs.head()

In [ ]:
dmrs['hypermethylated_samples'].value_counts(), \
dmrs['hypomethylated_samples'].value_counts()

In [ ]:
dmrs['size'] = dmrs['end'] - dmrs['start']
ax = sns.histplot(dmrs['size'])
ax.set_xlabel('DMR size', fontsize=16)
ax.set_ylabel('# DMRs', fontsize=16)
ax.set_title('DMR size distribution', fontsize=20)

In [ ]:
dms = 3
mask = dmrs['number_of_dms'] > dms
filt_dmrs = dmrs[mask]
filt_dmrs.shape

In [ ]:
filt_dmrs['size'] = filt_dmrs['end'] - filt_dmrs['start']
filt_dmrs['size'].mean()

In [ ]:
sns.histplot(filt_dmrs['size'])

In [ ]:
filt_dmrs.head()

In [ ]:
filt_dmrs['hypermethylated_samples'].value_counts(), \
filt_dmrs['hypomethylated_samples'].value_counts()

In [ ]:
tmp_hypo = filt_dmrs[filt_dmrs['hypomethylated_samples'] == f'IPS_{d}']
tmp_hyper = filt_dmrs[filt_dmrs['hypermethylated_samples'] == f'IPS_{d}']
tmp_hypo.shape, tmp_hyper.shape

### write DMRs out

In [ ]:
dmrs.shape

In [ ]:
dmrs['hypermethylated_samples'] = dmrs['hypermethylated_samples'].str.split(',')
dmrs['hypomethylated_samples'] = dmrs['hypomethylated_samples'].str.split(',')
dmrs = dmrs.explode('hypermethylated_samples').reset_index(drop=True)
dmrs = dmrs.explode('hypomethylated_samples').reset_index(drop=True)
dmrs.index = [f'{x}_{y}' for x,y in zip(dmrs['#chr'], dmrs['start'])]
dmrs.shape

In [ ]:
dmrs

In [ ]:
tmp_clusters = [f'IPS_main_{d}', f'IPS_alt_{d}', f'{d}_IPS']
outdir = f'{projdir}/csv/dmrs/mature_ips/by_group'

def write_dmrs(s):
    hypo_dmrs = dmrs[dmrs['hypomethylated_samples'] == s]
    hyper_dmrs = dmrs[dmrs['hypermethylated_samples'] == s]
    hypo_dmrs = hypo_dmrs[~hypo_dmrs.index.duplicated()]
    hyper_dmrs = hyper_dmrs[~hyper_dmrs.index.duplicated()]
    hypo_dmrs.to_csv(f'{outdir}/{s}_hypo.csv', sep='\t', header=None, index=None)
    hyper_dmrs.to_csv(f'{outdir}/{s}_hyper.csv', sep='\t', header=None, index=None)
    
    final_hypo = hypo_dmrs[hypo_dmrs['number_of_dms'] > 1]
    final_hyper = hyper_dmrs[hyper_dmrs['number_of_dms'] > 1]

    final_hypo.to_csv(f'{outdir}/{s}_hypo_dms1.csv', sep='\t', header=None, index=None)
    final_hyper.to_csv(f'{outdir}/{s}_hyper_dms1.csv', sep='\t', header=None, index=None)

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(write_dmrs, tmp_clusters), total=len(tmp_clusters)))

In [ ]:
for d in donors:
    indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
    dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/mature_ips/{d}/{d}_rms_results_collapsed.tsv',
                       header=0, sep='\t')
    ###
    tmp_clusters = [f'IPS_main_{d}', f'IPS_alt_{d}', f'{d}_IPS']
    outdir = f'{projdir}/csv/dmrs/mature_ips/by_group'
    with ProcessPoolExecutor() as executor:
        results = list(tqdm(executor.map(write_dmrs, tmp_clusters), total=len(tmp_clusters)))

### write allcools-friendly DMRs

In [ ]:
dmrs_list = []
for d in donors:
    indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
    dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/mature_ips/{d}/{d}_rms_results_collapsed.tsv',
                       header=0, sep='\t')
    dmrs = dmrs.iloc[:,:4]
    dmrs['donor'] = d
    dmrs_list.append(dmrs)
merged = pd.concat(dmrs_list)
merged.shape

In [ ]:
merged.head()

In [ ]:
dms = 1
mask = merged['number_of_dms'] > dms
filt_dmrs = merged[mask].copy()
filt_dmrs['chr_num'] = [int(x.split('chr')[1]) for x in filt_dmrs['#chr']]
filt_dmrs = filt_dmrs.sort_values(by=['chr_num', 'start'])
filt_dmrs.shape

In [ ]:
filt_dmrs

In [ ]:
out_dmrs = filt_dmrs.iloc[:,:3]
out = f'{projdir}/csv/dmrs/mature_ips/IPS_alt_hypo_dms1.bed'
out_dmrs.to_csv(out, sep='\t', header=None, index=None)
subprocess.run(['bgzip', '-f', out])
subprocess.run(['tabix', '-p', 'bed', f'{out}.gz'])
out_dmrs.to_csv(out, sep='\t', header=None, index=None)

### drop PMDs

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.shape

In [ ]:
dmrs_bt = BedTool.from_dataframe(filt_dmrs.iloc[:,:3])
pmds_bt = BedTool.from_dataframe(pmds)

In [ ]:
inter_dmrs = dmrs_bt.intersect(pmds_bt).to_dataframe()
inter_dmrs.shape[0], filt_dmrs.shape[0], inter_dmrs.shape[0] / filt_dmrs.shape[0]

In [ ]:
sub_dmrs = dmrs_bt.intersect(pmds_bt, v=True).to_dataframe()
sub_dmrs.shape[0], filt_dmrs.shape[0], sub_dmrs.shape[0] / filt_dmrs.shape[0]

In [ ]:
tmp_clusters = [f'IPS_main_{d}', f'IPS_alt_{d}', f'{d}_IPS']
outdir = f'{projdir}/csv/dmrs/mature_ips_drop_pmds/by_group'

def write_dmrs(s):
    hypo_dmrs = tmp_dmrs[tmp_dmrs['hypomethylated_samples'] == s]
    hyper_dmrs = tmp_dmrs[tmp_dmrs['hypermethylated_samples'] == s]
    hypo_dmrs = hypo_dmrs[~hypo_dmrs.index.duplicated()]
    hyper_dmrs = hyper_dmrs[~hyper_dmrs.index.duplicated()]
    hypo_dmrs.to_csv(f'{outdir}/{s}_hypo.csv', sep='\t', header=None, index=None)
    hyper_dmrs.to_csv(f'{outdir}/{s}_hyper.csv', sep='\t', header=None, index=None)
    
    final_hypo = hypo_dmrs[hypo_dmrs['number_of_dms'] > 1]
    final_hyper = hyper_dmrs[hyper_dmrs['number_of_dms'] > 1]

    final_hypo.to_csv(f'{outdir}/{s}_hypo_dms1.csv', sep='\t', header=None, index=None)
    final_hyper.to_csv(f'{outdir}/{s}_hyper_dms1.csv', sep='\t', header=None, index=None)

In [ ]:
for d in donors:
    indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
    dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/mature_ips/early_ips/{d}/{d}_rms_results_collapsed.tsv',
                       header=0, sep='\t')
    dmrs.index = [f'{x}_{y}' for x,y in zip(dmrs['#chr'],dmrs['start'])]
    dmrs_bt = BedTool.from_dataframe(dmrs.iloc[:,:3])
    pmds_bt = BedTool.from_dataframe(pmds)
    tmp = dmrs_bt.intersect(pmds_bt, v=True).to_dataframe()
    tmp.index = [f'{x}_{y}' for x,y in zip(tmp['chrom'],tmp['start'])]
    tmp_dmrs = dmrs.loc[tmp.index]
    ###
    tmp_clusters = [f'IPS_{d}', f'IPS_{d}', f'{d}_IPS']
    outdir = f'{projdir}/csv/dmrs/mature_ips_drop_pmds/early_ips/by_group'
    with ProcessPoolExecutor() as executor:
        results = list(tqdm(executor.map(write_dmrs, tmp_clusters), total=len(tmp_clusters)))

In [ ]:
!date